# Run an LLM on real phones — and find the right quant — with TinyEdge

Two calls against [TinyEdge](https://tinyedge.ai)'s device cloud:

1. **Benchmark** a GGUF model on real edge devices → decode tok/s, time-to-first-token, RAM, on-device perplexity.
2. The same call **`optimize=True`** → TinyEdge builds the quantization ladder from your f16, benchmarks every variant on every device as one sweep, and tells you *which quant to ship, per device* — measured, not guessed.

**Before you run:** a [tinyedge.ai](https://tinyedge.ai) account ($25 demo credit; this notebook uses ~$1) · paste your API key in the first code cell · at least one device paired via the TinyEdge Runner app with *"Available for benchmarks"* on · Kaggle **Internet enabled** (right sidebar; needs a phone-verified Kaggle account).

In [ ]:
%pip -q install "tinyedge>=0.2.4" huggingface_hub

In [ ]:
import tinyedge

# Paste your key from tinyedge.ai → New benchmark → "Your API key":
client = tinyedge.TinyEdge(api_key="tinyedge_sk_REPLACE_ME")

DEVICES = client.devices(online=True)     # devices that can claim a job right now
print("benchmarking on:", DEVICES)

**Grab a model and a quality corpus** — SmolLM2-135M (f16 source + a Q4 to benchmark as-is) and a small WikiText sample; jobs that carry a text corpus also measure perplexity on-device.

In [ ]:
# Models the standard way (huggingface_hub) — swap in any repo/file, or your own path.
from huggingface_hub import hf_hub_download
REPO = "bartowski/SmolLM2-135M-Instruct-GGUF"
Q4  = hf_hub_download(REPO, "SmolLM2-135M-Instruct-Q4_K_M.gguf")   # benchmark this one
F16 = hf_hub_download(REPO, "SmolLM2-135M-Instruct-f16.gguf")      # optimize from this one

# A WikiText sample for the on-device quality (perplexity) measurement.
import io, zipfile, pathlib, requests
pathlib.Path("corpus").mkdir(exist_ok=True)
wiki = zipfile.ZipFile(io.BytesIO(requests.get(
    "https://huggingface.co/datasets/ggml-org/ci/resolve/main/wikitext-2-raw-v1.zip", timeout=120).content))
pathlib.Path("corpus/wiki.txt").write_bytes(wiki.read("wikitext-2-raw/wiki.test.raw")[:200_000])
print("ready")

## Benchmark — one model, as-is

In [ ]:
# one result per device — decode tok/s, time-to-first-token, perplexity, RAM
client.benchmark(Q4, devices=DEVICES, dataset="corpus")

## Optimize — same call, one extra keyword
Generates the quant ladder from the f16, screens it, benchmarks everything as one sweep. (The f16 itself exceeds the current 250 MB upload cap — it's reported as excluded; its variants are what you'd ship anyway.)

In [ ]:
report = client.benchmark(F16, devices=DEVICES, dataset="corpus", optimize=True)
print(report.summary())
print("full report with charts + AI analysis:", report.sweep_url)